# 02 - Logistic Regression Model

This notebook trains a Logistic Regression baseline to classify student `performance_category` using the cleaned dataset from the preprocessing notebook.

The preprocessing and classifier are saved together as a scikit-learn pipeline in a `.pkl` file for later backend inference. Logistic Regression uses solver iterations rather than epochs.

In [ ]:
from pathlib import Path
import pickle

import pandas as pd
import numpy as np

from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

In [ ]:
# Project paths
PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == 'Notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA_DIR = PROJECT_ROOT / 'data'
PROCESSED_DIR = DATA_DIR / 'processed'
MODEL_DIR = PROJECT_ROOT / 'models'
MODEL_DIR.mkdir(exist_ok=True, parents=True)

DATA_PATH = PROCESSED_DIR / 'student_performance_processed.csv'
MODEL_PATH = MODEL_DIR / 'student_performance_model.pkl'

print(f'Loading dataset from: {DATA_PATH}')
df = pd.read_csv(DATA_PATH)
print(df.shape)
df.head()

In [ ]:
# Define target and feature columns
# Exclude outcome-derived fields to avoid target leakage.
target_col = 'performance_category'
id_col = 'student_id'

if target_col not in df.columns:
    raise ValueError(f'Target column {target_col} not found in the processed dataset.')

excluded_cols = [id_col, target_col, 'risk_level', 'final_marks', 'passed']
X = df.drop(columns=excluded_cols, errors='ignore').copy()
y = df[target_col].copy()

numeric_features = X.select_dtypes(include=[np.number]).columns.tolist()
categorical_features = X.select_dtypes(exclude=[np.number]).columns.tolist()

print('Numeric features:', numeric_features)
print('Categorical features:', categorical_features)

In [ ]:
# Train/test split
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y,
)

print(f'Train size: {len(X_train)}')
print(f'Test size: {len(X_test)}')

In [ ]:
# Build preprocessing and model pipeline
numeric_transformer = Pipeline(
    steps=[
        ('imputer', SimpleImputer(strategy='median')),
        ('scaler', StandardScaler()),
    ]
)

categorical_transformer = Pipeline(
    steps=[
        ('imputer', SimpleImputer(strategy='most_frequent')),
        ('onehot', OneHotEncoder(handle_unknown='ignore')),
    ]
)

preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, numeric_features),
        ('cat', categorical_transformer, categorical_features),
    ]
)

model = Pipeline(
    steps=[
        ('preprocessor', preprocessor),
        # Logistic regression uses solver iterations, not neural-network epochs.
        ('classifier', LogisticRegression(max_iter=1500)),
    ]
)

# Train the baseline model
model.fit(X_train, y_train)
y_pred = model.predict(X_test)

accuracy = accuracy_score(y_test, y_pred)
print(f'Baseline model accuracy: {accuracy:.4f}')
print('\nClassification report:')
print(classification_report(y_test, y_pred))
print('\nConfusion matrix:')
print(confusion_matrix(y_test, y_pred))

In [ ]:
# Save the trained model pipeline for backend use
with open(MODEL_PATH, 'wb') as model_file:
    pickle.dump(model, model_file)

print(f'Model saved to: {MODEL_PATH}')
print('This .pkl file can be loaded later by the backend for inference.')

## Conclusion

This notebook trains a Logistic Regression baseline to classify student `performance_category`. Outcome-derived fields such as `final_marks`, `passed`, and `risk_level` are excluded from the input features to reduce target leakage. The 1,500 setting is the solver's maximum iteration count, not an epoch count. Use the held-out test metrics above as a baseline when comparing future models, such as Random Forest or gradient boosting. The fitted preprocessing-and-classification pipeline is saved to `models/student_performance_model.pkl` for backend inference.